# MSLR 2026 - TWASEL

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import pickle, os, math, requests, random, copy
from collections import defaultdict
from typing import Optional, Tuple, List, Dict
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
class Config:
    DATA_PATH = "/path/to/data"
    NUM_KEYPOINTS = 86
    NUM_COORDS = 2
    INPUT_DIM = 172

    # Region keypoint ranges
    BODY_KP_START, BODY_KP_END = 0, 33
    LHAND_KP_START, LHAND_KP_END = 33, 54
    RHAND_KP_START, RHAND_KP_END = 54, 75
    FACE_KP_START, FACE_KP_END = 75, 86

    # Flattened feature slices
    BODY_FEAT_START, BODY_FEAT_END = 0, 66
    LHAND_FEAT_START, LHAND_FEAT_END = 66, 108
    RHAND_FEAT_START, RHAND_FEAT_END = 108, 150
    FACE_FEAT_START, FACE_FEAT_END = 150, 172
    BODY_DIM, LHAND_DIM, RHAND_DIM, FACE_DIM = 66, 42, 42, 22

    # Multi-Stream Encoder (DEEPER: 6 blocks, was 4)
    D_STREAM = 128
    D_MODEL = 256
    NUM_HEADS = 8
    NUM_CONFORMER_BLOCKS = 6
    CONV_KERNEL_SIZE = 31
    FF_EXPANSION_FACTOR = 4
    DROPOUT = 0.15
    CONV_DROPOUT = 0.1
    STOCHASTIC_DEPTH_PROB = 0.15
    USE_VELOCITY = True

    # Decoder (DEEPER: 3 layers, was 2)
    NUM_DECODER_LAYERS = 3
    DECODER_HEADS = 8
    DECODER_FF_DIM = 1024
    DECODER_DROPOUT = 0.15
    MAX_DECODE_LEN = 60

    # Hybrid loss - Stage 1
    CTC_WEIGHT = 0.3
    LABEL_SMOOTHING = 0.1

    # Training - Stage 1
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    LEARNING_RATE = 3e-4
    MIN_LR = 1e-6
    WEIGHT_DECAY = 0.01
    WARMUP_EPOCHS = 5
    GRADIENT_CLIP = 1.0
    USE_AMP = False

    # Training - Stage 2 (Fine-tuning)
    STAGE2_EPOCHS = 15
    STAGE2_LR = 5e-5
    STAGE2_MIN_LR = 1e-6
    STAGE2_CTC_WEIGHT = 0.1       # Reduce CTC (it overfits 3.5x)
    STAGE2_LABEL_SMOOTHING = 0.1
    STAGE2_WARMUP_EPOCHS = 2
    STAGE2_AUG_SCALE = 0.5        # Reduce augmentation intensity

    # Augmentation
    AUG_ROTATION_RANGE = 20
    AUG_SCALE_RANGE = (0.85, 1.15)
    AUG_SHIFT_RANGE = 0.15
    AUG_TIME_WARP_RANGE = (0.8, 1.2)
    AUG_TEMPORAL_MASK_PROB = 0.15
    AUG_TEMPORAL_MASK_LENGTH = 10
    AUG_SPATIAL_MASK_PROB = 0.15
    AUG_SPATIAL_MASK_NUM = 8
    AUG_NOISE_STD = 0.02
    AUG_FLIP_PROB = 0.3

    # Normalization reference keypoints
    LEFT_SHOULDER_IDX = 11
    RIGHT_SHOULDER_IDX = 12
    BODY_SWAP_PAIRS = [
        (11,12),(13,14),(15,16),(17,18),(19,20),
        (21,22),(23,24),(25,26),(27,28),(29,30),(31,32),
    ]

    # CTC decoding
    BEAM_WIDTH = 10
    PATIENCE = 10

    # AR Beam Search (WIDER: 10, was 5)
    AR_BEAM_WIDTH = 10
    LENGTH_PENALTY = 0.6  # Default, will be swept

    # Length Penalty Sweep
    LP_SWEEP_VALUES = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]


config = Config()
print(f"Config: D_STREAM={config.D_STREAM}, D_MODEL={config.D_MODEL}, Conformer x{config.NUM_CONFORMER_BLOCKS}")
print(f"Decoder: {config.NUM_DECODER_LAYERS} layers, CTC_WEIGHT={config.CTC_WEIGHT}")
print(f"Stage 2: {config.STAGE2_EPOCHS} epochs, LR={config.STAGE2_LR}, CTC_WEIGHT={config.STAGE2_CTC_WEIGHT}")
print(f"AR Beam: width={config.AR_BEAM_WIDTH}, LP sweep={config.LP_SWEEP_VALUES}")

## 3. Data Loading

In [ ]:
def download_annotations():
    base_url = "https://raw.githubusercontent.com/gufranSabri/Pose86K-CSLR-Isharah/main/annotations_v2/isharah2000/SI"
    for split in ["train", "dev"]:
        url = f"{base_url}/{split}.csv"
        r = requests.get(url)
        if r.status_code == 200:
            with open(f"{split}.csv", "wb") as f: f.write(r.content)
            print(f"Downloaded {split}.csv")
        else: raise RuntimeError(f"Failed: {r.status_code}")

download_annotations()
print(f"\nLoading {config.DATA_PATH}...")
pose_data = pd.read_pickle(config.DATA_PATH)
train_labels = pd.read_csv("train.csv", sep="|")
dev_labels = pd.read_csv("dev.csv", sep="|")
print(f"Train: {len(train_labels)}, Dev: {len(dev_labels)}, Total: {len(pose_data)}")
assert len(train_labels) + len(dev_labels) == len(pose_data)

## 4. Vocabulary (with SOS/EOS)

In [ ]:
def build_vocabulary(train_df, dev_df):
    vocab = set()
    for seq in train_df["gloss"].tolist() + dev_df["gloss"].tolist():
        vocab.update(seq.replace(" _", "_").split())
    sorted_v = sorted(vocab)
    w2i = {w: i+1 for i, w in enumerate(sorted_v)}
    i2w = {i: w for w, i in w2i.items()}
    i2w[0] = "<blank>"
    return w2i, i2w

word2idx, idx2word = build_vocabulary(train_labels, dev_labels)
NUM_CLASSES = len(word2idx) + 1
SOS_IDX = NUM_CLASSES
EOS_IDX = NUM_CLASSES + 1
DECODER_VOCAB_SIZE = NUM_CLASSES + 2
idx2word[SOS_IDX] = "<sos>"
idx2word[EOS_IDX] = "<eos>"
print(f"Vocab: {len(word2idx)} glosses, CTC classes: {NUM_CLASSES}, Decoder vocab: {DECODER_VOCAB_SIZE}")

## 5. Normalization + Augmentation

In [ ]:
class PoseNormalizer:
    def __init__(self, cfg):
        self.ls = cfg.LEFT_SHOULDER_IDX
        self.rs = cfg.RIGHT_SHOULDER_IDX

    def __call__(self, kp):
        kp = kp.copy().astype(np.float32)
        kp = np.where(np.isinf(kp), 0.0, kp)
        kp = self._interp(kp)
        center = (kp[:, self.ls, :] + kp[:, self.rs, :]) / 2
        diff = kp[:, self.rs, :] - kp[:, self.ls, :]
        widths = np.sqrt(np.sum(diff**2, axis=-1, keepdims=True))
        mw = max(np.median(widths), 0.01)
        kp = (kp - center[:, np.newaxis, :]) / mw
        kp = np.nan_to_num(kp, nan=0.0, posinf=0.0, neginf=0.0)
        return np.clip(kp, -10.0, 10.0)

    def _interp(self, kp):
        for k in range(kp.shape[1]):
            for c in range(kp.shape[2]):
                v = kp[:, k, c]; m = np.isnan(v)
                if m.any() and not m.all():
                    vi = np.where(~m)[0]; ii = np.where(m)[0]
                    kp[ii, k, c] = np.interp(ii, vi, v[vi])
                elif m.all(): kp[:, k, c] = 0.0
        return kp


class PoseAugmentor:
    def __init__(self, cfg, scale=1.0):
        """scale: multiplier for augmentation intensity (0.5 = half strength for fine-tuning)."""
        self.rot = cfg.AUG_ROTATION_RANGE * scale
        self.sc = (1 - (1 - cfg.AUG_SCALE_RANGE[0]) * scale, 1 + (cfg.AUG_SCALE_RANGE[1] - 1) * scale)
        self.sh = cfg.AUG_SHIFT_RANGE * scale
        self.tw = (1 - (1 - cfg.AUG_TIME_WARP_RANGE[0]) * scale, 1 + (cfg.AUG_TIME_WARP_RANGE[1] - 1) * scale)
        self.tm_prob = cfg.AUG_TEMPORAL_MASK_PROB * scale
        self.tm_len = cfg.AUG_TEMPORAL_MASK_LENGTH
        self.sm_prob = cfg.AUG_SPATIAL_MASK_PROB * scale
        self.sm_num = cfg.AUG_SPATIAL_MASK_NUM
        self.noise = cfg.AUG_NOISE_STD * scale
        self.flip_prob = cfg.AUG_FLIP_PROB  # Keep flip probability unchanged
        self.swap_pairs = cfg.BODY_SWAP_PAIRS
        self.lh_s, self.lh_e = cfg.LHAND_KP_START, cfg.LHAND_KP_END
        self.rh_s, self.rh_e = cfg.RHAND_KP_START, cfg.RHAND_KP_END

    def __call__(self, kp):
        kp = kp.copy()
        kp = self._rotate(kp); kp = self._scale(kp); kp = self._shift(kp)
        kp = self._flip(kp); kp = self._add_noise(kp)
        kp = self._time_warp(kp); kp = self._temporal_mask(kp); kp = self._spatial_mask(kp)
        return kp

    def _rotate(self, kp):
        a = np.deg2rad(np.random.uniform(-self.rot, self.rot))
        R = np.array([[np.cos(a),-np.sin(a)],[np.sin(a),np.cos(a)]], dtype=np.float32)
        return kp @ R

    def _scale(self, kp): return kp * np.random.uniform(*self.sc)

    def _shift(self, kp):
        kp[:,:,0] += np.random.uniform(-self.sh, self.sh)
        kp[:,:,1] += np.random.uniform(-self.sh, self.sh)
        return kp

    def _flip(self, kp):
        if np.random.random() >= self.flip_prob: return kp
        kp[:,:,0] = -kp[:,:,0]
        for l, r in self.swap_pairs: kp[:,[l,r],:] = kp[:,[r,l],:]
        lh = kp[:,self.lh_s:self.lh_e,:].copy(); rh = kp[:,self.rh_s:self.rh_e,:].copy()
        kp[:,self.lh_s:self.lh_e,:] = rh; kp[:,self.rh_s:self.rh_e,:] = lh
        return kp

    def _add_noise(self, kp):
        return kp + np.random.normal(0, self.noise, kp.shape).astype(np.float32) if self.noise > 0 else kp

    def _time_warp(self, kp):
        T = len(kp)
        if T < 10: return kp
        nT = max(10, int(T * np.random.uniform(*self.tw)))
        oi = np.linspace(0, T-1, T); ni = np.linspace(0, T-1, nT)
        w = np.zeros((nT, kp.shape[1], kp.shape[2]), dtype=np.float32)
        for k in range(kp.shape[1]):
            for c in range(kp.shape[2]): w[:,k,c] = np.interp(ni, oi, kp[:,k,c])
        return w

    def _temporal_mask(self, kp):
        if np.random.random() > self.tm_prob: return kp
        T = len(kp); ml = min(self.tm_len, T//4)
        if ml < 1: return kp
        s = np.random.randint(0, max(1, T-ml)); kp[s:s+ml] = 0.0; return kp

    def _spatial_mask(self, kp):
        if np.random.random() > self.sm_prob: return kp
        idx = np.random.choice(kp.shape[1], min(self.sm_num, kp.shape[1]//4), replace=False)
        kp[:,idx,:] = 0.0; return kp


normalizer = PoseNormalizer(config)
augmentor = PoseAugmentor(config, scale=1.0)
augmentor_mild = PoseAugmentor(config, scale=config.STAGE2_AUG_SCALE)

## 6. Dataset

In [ ]:
class SignLanguageDataset(Dataset):
    def __init__(self, pose_data, annot, w2i, norm, aug=None, train=True):
        self.pose_data = pose_data; self.annot = annot.reset_index(drop=True)
        self.w2i = w2i; self.norm = norm; self.aug = aug if train else None
    def __len__(self): return len(self.annot)
    def __getitem__(self, idx):
        s = self.annot.iloc[idx]; sid = s["id"]
        kp = np.array(self.pose_data[sid]["keypoints"], dtype=np.float32)
        kp = self.norm(kp)
        if self.aug: kp = self.aug(kp)
        kp = np.nan_to_num(kp.reshape(kp.shape[0], -1), nan=0., posinf=0., neginf=0.)
        tgt = [self.w2i[g] for g in s["gloss"].replace(" _","_").split()]
        return torch.tensor(kp, dtype=torch.float32), torch.tensor(tgt, dtype=torch.long), sid

def collate_fn(batch):
    xs = [b[0] for b in batch]; ys = [b[1] for b in batch]; ids = [b[2] for b in batch]
    xl = torch.tensor([len(x) for x in xs], dtype=torch.long)
    yl = torch.tensor([len(y) for y in ys], dtype=torch.long)
    return pad_sequence(xs, True, 0.), pad_sequence(ys, True, 0), xl, yl, ids

train_dataset = SignLanguageDataset(pose_data, train_labels, word2idx, normalizer, augmentor, True)
val_dataset = SignLanguageDataset(pose_data, dev_labels, word2idx, normalizer, None, False)

print("Validating samples...")
bad = sum(1 for i in range(min(500, len(train_dataset))) if torch.isnan(train_dataset[i][0]).any())
print(f"NaN samples: {bad}")

train_loader = DataLoader(train_dataset, config.BATCH_SIZE, True, collate_fn=collate_fn, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, config.BATCH_SIZE, False, collate_fn=collate_fn, num_workers=0, pin_memory=True)
print(f"Train: {len(train_dataset)} ({len(train_loader)} batches), Val: {len(val_dataset)} ({len(val_loader)} batches)")

## 7. Model Building Blocks

In [ ]:
class DropPath(nn.Module):
    def __init__(self, p=0.): super().__init__(); self.p = p
    def forward(self, x):
        if self.p == 0. or not self.training: return x
        k = 1 - self.p; m = (torch.rand((x.shape[0],)+(1,)*(x.ndim-1), dtype=x.dtype, device=x.device)+k).floor_()
        return x / k * m

class Swish(nn.Module):
    def forward(self, x): return x * torch.sigmoid(x)

class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=5000, drop=0.1):
        super().__init__(); self.drop = nn.Dropout(drop)
        pe = torch.zeros(max_len, d); pos = torch.arange(max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.) / d))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return self.drop(x + self.pe[:, :x.size(1)])

class ConvolutionModule(nn.Module):
    def __init__(self, d, ks=31, drop=0.1):
        super().__init__()
        self.ln = nn.LayerNorm(d); self.pw1 = nn.Conv1d(d, 2*d, 1); self.glu = nn.GLU(1)
        self.dw = nn.Conv1d(d, d, ks, padding=ks//2, groups=d)
        self.bn = nn.BatchNorm1d(d); self.act = Swish(); self.pw2 = nn.Conv1d(d, d, 1); self.do = nn.Dropout(drop)
    def forward(self, x):
        x = self.ln(x).transpose(1,2)
        x = self.act(self.bn(self.dw(self.glu(self.pw1(x)))))
        return self.do(self.pw2(x)).transpose(1,2)

class FeedForwardModule(nn.Module):
    def __init__(self, d, exp=4, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d*exp), Swish(), nn.Dropout(drop), nn.Linear(d*exp, d), nn.Dropout(drop))
    def forward(self, x): return self.net(x)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d, h, drop=0.1):
        super().__init__()
        self.ln = nn.LayerNorm(d); self.attn = nn.MultiheadAttention(d, h, dropout=drop, batch_first=True); self.do = nn.Dropout(drop)
    def forward(self, x, mask=None):
        x = self.ln(x); o, _ = self.attn(x, x, x, key_padding_mask=mask, need_weights=False); return self.do(o)

class ConformerBlock(nn.Module):
    def __init__(self, d, h, ks=31, exp=4, drop=0.1, cdrop=0.1, dp=0.):
        super().__init__()
        self.ff1 = FeedForwardModule(d, exp, drop); self.mhsa = MultiHeadSelfAttention(d, h, drop)
        self.conv = ConvolutionModule(d, ks, cdrop); self.ff2 = FeedForwardModule(d, exp, drop)
        self.ln = nn.LayerNorm(d); self.dp = DropPath(dp) if dp > 0 else nn.Identity()
    def forward(self, x, mask=None):
        x = x + 0.5*self.dp(self.ff1(x)); x = x + self.dp(self.mhsa(x, mask))
        x = x + self.dp(self.conv(x)); x = x + 0.5*self.dp(self.ff2(x)); return self.ln(x)

## 8. Multi-Stream Encoders + Fusion

In [ ]:
class RegionEncoder(nn.Module):
    def __init__(self, in_d, d_s, drop=0.1):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_d, d_s), nn.LayerNorm(d_s), nn.ReLU(), nn.Dropout(drop))
        self.conv = nn.Sequential(
            nn.Conv1d(d_s, d_s, 3, padding=1), nn.BatchNorm1d(d_s), nn.ReLU(), nn.Dropout(drop),
            nn.Conv1d(d_s, d_s, 3, padding=1), nn.BatchNorm1d(d_s), nn.ReLU(), nn.Dropout(drop),
        )
    def forward(self, x):
        x = self.proj(x); return self.conv(x.transpose(1,2)).transpose(1,2)

class GatedFusion(nn.Module):
    def __init__(self, n, d_s, d_m):
        super().__init__()
        self.projs = nn.ModuleList([nn.Linear(d_s, d_m) for _ in range(n)])
        self.gate = nn.Sequential(nn.Linear(d_m*n, d_m), nn.ReLU(), nn.Linear(d_m, n))
        self.ln = nn.LayerNorm(d_m)
    def forward(self, streams):
        p = [proj(s) for proj, s in zip(self.projs, streams)]
        g = F.softmax(self.gate(torch.cat(p, -1)), -1)  # (B,T,N)
        return self.ln((torch.stack(p, -1) * g.unsqueeze(2)).sum(-1))

## 9. Decoder + Full Model

In [ ]:
class GlossDecoder(nn.Module):
    def __init__(self, voc, d, h, nl, ff, drop, max_len=300):
        super().__init__()
        self.emb = nn.Embedding(voc, d); self.pe = PositionalEncoding(d, max_len, drop); self.sc = math.sqrt(d)
        layer = nn.TransformerDecoderLayer(d, h, ff, drop, batch_first=True, norm_first=True)
        self.dec = nn.TransformerDecoder(layer, nl)
        self.out = nn.Linear(d, voc)
    def forward(self, tgt, mem, tgt_mask=None, tgt_pad=None, mem_pad=None):
        return self.out(self.dec(self.pe(self.emb(tgt)*self.sc), mem, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_pad, memory_key_padding_mask=mem_pad))


class MultiStreamHybridCSLR(nn.Module):
    def __init__(self, nc, dv, cfg):
        super().__init__()
        ds = cfg.D_STREAM; dm = cfg.D_MODEL; v = 2 if cfg.USE_VELOCITY else 1
        self.body_enc = RegionEncoder(cfg.BODY_DIM*v, ds, cfg.DROPOUT)
        self.lh_enc = RegionEncoder(cfg.LHAND_DIM*v, ds, cfg.DROPOUT)
        self.rh_enc = RegionEncoder(cfg.RHAND_DIM*v, ds, cfg.DROPOUT)
        self.face_enc = RegionEncoder(cfg.FACE_DIM*v, ds, cfg.DROPOUT)
        self.fusion = GatedFusion(4, ds, dm)
        self.pe = PositionalEncoding(dm, drop=cfg.DROPOUT)
        nb = cfg.NUM_CONFORMER_BLOCKS
        dp = [cfg.STOCHASTIC_DEPTH_PROB*i/max(nb-1,1) for i in range(nb)]
        self.blocks = nn.ModuleList([ConformerBlock(dm, cfg.NUM_HEADS, cfg.CONV_KERNEL_SIZE, cfg.FF_EXPANSION_FACTOR, cfg.DROPOUT, cfg.CONV_DROPOUT, dp[i]) for i in range(nb)])
        self.ctc_head = nn.Sequential(nn.Linear(dm, dm), nn.ReLU(), nn.Dropout(cfg.DROPOUT), nn.Linear(dm, nc))
        self.decoder = GlossDecoder(dv, dm, cfg.DECODER_HEADS, cfg.NUM_DECODER_LAYERS, cfg.DECODER_FF_DIM, cfg.DECODER_DROPOUT)
        self.bs = slice(cfg.BODY_FEAT_START, cfg.BODY_FEAT_END)
        self.ls = slice(cfg.LHAND_FEAT_START, cfg.LHAND_FEAT_END)
        self.rs = slice(cfg.RHAND_FEAT_START, cfg.RHAND_FEAT_END)
        self.fs = slice(cfg.FACE_FEAT_START, cfg.FACE_FEAT_END)
        self.use_vel = cfg.USE_VELOCITY

    @staticmethod
    def _vel(x):
        v = torch.zeros_like(x); v[:,1:] = x[:,1:] - x[:,:-1]; return v

    def encode(self, x, lengths):
        B, T, _ = x.shape
        body, lh, rh, face = x[:,:,self.bs], x[:,:,self.ls], x[:,:,self.rs], x[:,:,self.fs]
        if self.use_vel:
            body = torch.cat([body, self._vel(body)],-1); lh = torch.cat([lh, self._vel(lh)],-1)
            rh = torch.cat([rh, self._vel(rh)],-1); face = torch.cat([face, self._vel(face)],-1)
        fused = self.fusion([self.body_enc(body), self.lh_enc(lh), self.rh_enc(rh), self.face_enc(face)])
        fused = self.pe(fused)
        mask = torch.arange(T, device=x.device).unsqueeze(0).expand(B,T) >= lengths.unsqueeze(1).to(x.device)
        for blk in self.blocks: fused = blk(fused, mask)
        return fused, mask

    def forward(self, x, lengths, dec_in=None, dec_lens=None):
        enc, emask = self.encode(x, lengths)
        ctc = self.ctc_head(enc)
        dec_out = None
        if dec_in is not None:
            S = dec_in.size(1)
            tmask = nn.Transformer.generate_square_subsequent_mask(S, device=x.device)
            tpad = None
            if dec_lens is not None:
                tpad = torch.arange(S, device=x.device).unsqueeze(0).expand(x.size(0),S) >= dec_lens.unsqueeze(1).to(x.device)
            dec_out = self.decoder(dec_in, enc, tmask, tpad, emask)
        return ctc, dec_out, enc


model = MultiStreamHybridCSLR(NUM_CLASSES, DECODER_VOCAB_SIZE, config).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Quick shape test
with torch.no_grad():
    dx = torch.randn(2,50,172, device=device); dl = torch.tensor([50,30], device=device)
    dd = torch.randint(0,10,(2,5), device=device); ddl = torch.tensor([5,4], device=device)
    c,d,e = model(dx, dl, dd, ddl)
    print(f"CTC: {c.shape}, Dec: {d.shape}, Enc: {e.shape} â€” OK")

## 10. Decoding Functions (CTC + AR Greedy + AR Beam Search)

In [ ]:
def greedy_decode_ctc(log_probs, idx2word):
    decoded, prev = [], -1
    for t in log_probs.argmax(-1):
        t = t.item()
        if t != 0 and t != prev: decoded.append(idx2word.get(t, '<unk>'))
        prev = t
    return decoded


def beam_search_ctc(log_probs, idx2word, bw=10, blank=0):
    T, V = log_probs.shape; lp = log_probs.cpu().numpy()
    beams = {(): (0., float('-inf'))}
    for t in range(T):
        nb = defaultdict(lambda: (float('-inf'), float('-inf')))
        topk = np.argsort(lp[t])[-bw*2:]
        for pf, (pb, pnb) in beams.items():
            pt = np.logaddexp(pb, pnb)
            for c in topk:
                pc = lp[t,c]
                if c == blank:
                    o = nb[pf]; nb[pf] = (np.logaddexp(o[0], pt+pc), o[1])
                else:
                    np_ = pf + (c,)
                    if pf and pf[-1] == c:
                        o = nb[np_]; nb[np_] = (o[0], np.logaddexp(o[1], pb+pc))
                        o2 = nb[pf]; nb[pf] = (o2[0], np.logaddexp(o2[1], pnb+pc))
                    else:
                        o = nb[np_]; nb[np_] = (o[0], np.logaddexp(o[1], pt+pc))
        sc = [(p, a, b, np.logaddexp(a,b)) for p,(a,b) in nb.items()]
        sc.sort(key=lambda x: x[3], reverse=True)
        beams = {p: (a,b) for p,a,b,_ in sc[:bw]}
    best = max(beams.keys(), key=lambda p: np.logaddexp(*beams[p]))
    return [idx2word.get(i,'<unk>') for i in best]


@torch.no_grad()
def ar_greedy_decode(model, enc_out, enc_mask, idx2word, sos, eos, max_len=60):
    """Autoregressive greedy decoding."""
    model.eval(); B = enc_out.size(0); dev = enc_out.device
    gen = torch.full((B,1), sos, dtype=torch.long, device=dev)
    done = torch.zeros(B, dtype=torch.bool, device=dev)
    for _ in range(max_len):
        S = gen.size(1)
        tm = nn.Transformer.generate_square_subsequent_mask(S, device=dev)
        logits = model.decoder(gen, enc_out, tm, None, enc_mask)
        nxt = logits[:,-1,:].argmax(-1).masked_fill(done, eos)
        gen = torch.cat([gen, nxt.unsqueeze(1)], 1)
        done = done | (nxt == eos)
        if done.all(): break
    results = []
    for i in range(B):
        w = [idx2word.get(t,'<unk>') for t in gen[i].tolist() if t not in (sos, eos, 0) and idx2word.get(t,'') not in ('<blank>','<sos>','<eos>')]
        results.append(w)
    return results

def _lp_score(seq_len, score, alpha=0.6):
    """Google NMT length penalty: ((5+len)/(5+1))^alpha."""
    if seq_len <= 0: return score
    return score / ((5.0 + seq_len) / 6.0) ** alpha


@torch.no_grad()
def ar_beam_search(model, encoder_output, enc_padding_mask,
                   idx2word, sos_idx, eos_idx,
                   beam_width=5, max_len=60, length_penalty=0.6):
    """Autoregressive beam search decoding with length normalization.
    
    Processes one sample at a time (B=1 per beam step) to manage memory.
    """
    model.eval()
    B = encoder_output.size(0)
    dev = encoder_output.device
    all_results = []

    for b in range(B):
        enc_b = encoder_output[b:b+1]       # (1, T, D)
        mask_b = enc_padding_mask[b:b+1] if enc_padding_mask is not None else None

        # Each beam: (token_list, cumulative_log_prob)
        beams = [([sos_idx], 0.0)]
        completed = []

        for step in range(max_len):
            if not beams:
                break

            candidates = []
            for seq, score in beams:
                if seq[-1] == eos_idx:
                    completed.append((seq, score))
                    continue

                tgt = torch.tensor([seq], dtype=torch.long, device=dev)
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(len(seq), device=dev)

                logits = model.decoder(tgt, enc_b, tgt_mask, None, mask_b)
                log_probs = F.log_softmax(logits[0, -1], dim=-1)  # (vocab,)

                topk_lp, topk_id = log_probs.topk(beam_width)
                for i in range(beam_width):
                    tok = topk_id[i].item()
                    candidates.append((seq + [tok], score + topk_lp[i].item()))

            if not candidates:
                break

            # Rank by length-penalized score
            candidates.sort(
                key=lambda x: _lp_score(len(x[0])-1, x[1], length_penalty),
                reverse=True,
            )
            beams = candidates[:beam_width]

            # Early stop if all beams ended
            if all(bm[0][-1] == eos_idx for bm in beams):
                completed.extend(beams)
                break

        # Finalize: merge remaining beams with completed
        completed.extend(beams)
        if completed:
            best_seq, _ = max(
                completed,
                key=lambda x: _lp_score(len(x[0])-1, x[1], length_penalty),
            )
        else:
            best_seq = [sos_idx]

        # Convert to words, skip special tokens
        words = []
        for t in best_seq:
            if t == sos_idx: continue
            if t == eos_idx: break
            w = idx2word.get(t, '<unk>')
            if w not in ('<blank>', '<sos>', '<eos>'):
                words.append(w)
        all_results.append(words)

    return all_results

## 11. Metrics + LR Scheduler

In [ ]:
def levenshtein(ref, hyp):
    m, n = len(ref), len(hyp)
    if m == 0: return n
    if n == 0: return m
    dp = np.zeros((m+1, n+1), dtype=int)
    for i in range(m+1): dp[i,0] = i
    for j in range(n+1): dp[0,j] = j
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i,j] = min(dp[i-1,j]+1, dp[i,j-1]+1, dp[i-1,j-1]+(0 if ref[i-1]==hyp[j-1] else 1))
    return dp[m,n]

def calc_wer(refs, hyps):
    td = tw = 0
    for r, h in zip(refs, hyps): td += levenshtein(r, h); tw += len(r)
    return td / max(tw, 1), td, tw

class WarmupCosineScheduler:
    def __init__(self, opt, we, te, mlr=1e-6, blr=None):
        self.opt=opt; self.we=we; self.te=te; self.mlr=mlr; self.blr=blr or opt.param_groups[0]['lr']; self.ep=0
    def step(self):
        self.ep += 1; lr = self.get_lr()
        for pg in self.opt.param_groups: pg['lr'] = lr
    def get_lr(self):
        if self.ep <= self.we: return self.blr * self.ep / self.we
        p = (self.ep - self.we) / (self.te - self.we)
        return self.mlr + 0.5 * (self.blr - self.mlr) * (1 + math.cos(math.pi * p))

## 12. Hybrid Trainer

In [ ]:
class HybridTrainer:
    def __init__(self, model, tl, vl, cfg, i2w, sos, eos, dvs, ctc_weight=None, lr=None, min_lr=None, warmup=None, num_epochs=None):
        self.model=model; self.tl=tl; self.vl=vl; self.cfg=cfg; self.i2w=i2w
        self.sos=sos; self.eos=eos; self.dvs=dvs
        self.ctc_w = ctc_weight if ctc_weight is not None else cfg.CTC_WEIGHT
        self.ctc_crit = nn.CTCLoss(blank=0, zero_infinity=True)
        self.ce_crit = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=cfg.LABEL_SMOOTHING)
        _lr = lr if lr is not None else cfg.LEARNING_RATE
        self.opt = optim.AdamW(model.parameters(), lr=_lr, weight_decay=cfg.WEIGHT_DECAY, eps=1e-8)
        _min = min_lr if min_lr is not None else cfg.MIN_LR
        _warm = warmup if warmup is not None else cfg.WARMUP_EPOCHS
        _total = num_epochs if num_epochs is not None else cfg.NUM_EPOCHS
        self.sched = WarmupCosineScheduler(self.opt, _warm, _total, _min, _lr)
        self.hist = {k: [] for k in ['train_loss','train_ctc','train_ce','train_wer','val_loss','val_ctc','val_ce','val_wer','lr']}
        self.best_wer = float('inf'); self.patience = 0

    def _prep_dec(self, tgt, tl):
        B = tgt.size(0); S = tl.max().item() + 1
        di = torch.full((B,S), 0, dtype=torch.long, device=device)
        dt = torch.full((B,S), -100, dtype=torch.long, device=device)
        dl = torch.zeros(B, dtype=torch.long, device=device)
        for i in range(B):
            n = tl[i].item()
            di[i,0] = self.sos; di[i,1:n+1] = tgt[i,:n]
            dt[i,:n] = tgt[i,:n]; dt[i,n] = self.eos
            dl[i] = n + 1
        return di, dt, dl

    def train_epoch(self):
        self.model.train()
        tl_=tc_=te_=0.; nb=0; refs=[]; hyps=[]
        for inp, tgt, il, tl, _ in tqdm(self.tl, desc='Train'):
            inp=inp.to(device); tgt=tgt.to(device); il=il.to(device); tl=tl.to(device)
            if torch.isnan(inp).any(): continue
            di, dt, dl = self._prep_dec(tgt, tl)
            self.opt.zero_grad()
            ctc_l, dec_l, _ = self.model(inp, il, di, dl)
            clp = F.log_softmax(ctc_l, -1)
            if torch.isnan(clp).any(): continue
            c_loss = self.ctc_crit(clp.permute(1,0,2), tgt, il, tl)
            e_loss = self.ce_crit(dec_l.reshape(-1, self.dvs), dt.reshape(-1))
            loss = self.ctc_w * c_loss + (1 - self.ctc_w) * e_loss
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.GRADIENT_CLIP)
            self.opt.step()
            tl_ += loss.item(); tc_ += c_loss.item(); te_ += e_loss.item(); nb += 1
            if len(refs) < 500:
                with torch.no_grad():
                    for j in range(min(4, len(inp))):
                        hyps.append(greedy_decode_ctc(clp[j,:il[j].item()], self.i2w))
                        refs.append([self.i2w[t.item()] for t in tgt[j,:tl[j].item()]])
        w,_,_ = calc_wer(refs, hyps)
        return tl_/max(nb,1), tc_/max(nb,1), te_/max(nb,1), w

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        tl_=tc_=te_=0.; nb=0; refs=[]; hyps=[]
        for inp, tgt, il, tl, _ in tqdm(self.vl, desc='Val'):
            inp=inp.to(device); tgt=tgt.to(device); il=il.to(device); tl=tl.to(device)
            if torch.isnan(inp).any(): continue
            di, dt, dl = self._prep_dec(tgt, tl)
            ctc_l, dec_l, _ = self.model(inp, il, di, dl)
            clp = F.log_softmax(ctc_l, -1)
            if torch.isnan(clp).any(): continue
            c_loss = self.ctc_crit(clp.permute(1,0,2), tgt, il, tl)
            e_loss = self.ce_crit(dec_l.reshape(-1, self.dvs), dt.reshape(-1))
            loss = self.ctc_w * c_loss + (1 - self.ctc_w) * e_loss
            if not torch.isnan(loss):
                tl_ += loss.item(); tc_ += c_loss.item(); te_ += e_loss.item(); nb += 1
            for j in range(len(inp)):
                hyps.append(greedy_decode_ctc(clp[j,:il[j].item()], self.i2w))
                refs.append([self.i2w[t.item()] for t in tgt[j,:tl[j].item()]])
        w,_,_ = calc_wer(refs, hyps)
        return tl_/max(nb,1), tc_/max(nb,1), te_/max(nb,1), w

    def train(self, num_epochs, stage_name=""):
        prefix = f"[{stage_name}] " if stage_name else ""
        print(f"{prefix}Training {num_epochs} epochs | Loss: {self.ctc_w}*CTC + {1-self.ctc_w:.1f}*CE")
        print('-'*70)
        for ep in range(1, num_epochs+1):
            lr = self.opt.param_groups[0]['lr']
            print(f'\n{prefix}Epoch {ep}/{num_epochs} | LR: {lr:.6f}')
            tl, tc, te, tw = self.train_epoch()
            vl, vc, ve, vw = self.validate()
            self.sched.step()
            for k,v in zip(['train_loss','train_ctc','train_ce','train_wer','val_loss','val_ctc','val_ce','val_wer','lr'],[tl,tc,te,tw,vl,vc,ve,vw,lr]):
                self.hist[k].append(v)
            print(f'Train Loss:{tl:.4f} (CTC={tc:.4f} CE={te:.4f}) WER:{tw:.2%}')
            print(f'Val   Loss:{vl:.4f} (CTC={vc:.4f} CE={ve:.4f}) WER:{vw:.2%}')

            if vw < self.best_wer:
                self.best_wer = vw; self.patience = 0
                torch.save({'epoch':ep, 'model_state_dict':self.model.state_dict(),
                            'optimizer_state_dict':self.opt.state_dict(), 'val_wer':vw,
                            'stage': stage_name}, 'best_model.pth')
                print(f'*** Best model! WER: {vw:.2%} ***')
            else:
                self.patience += 1
                print(f'No improvement. Patience: {self.patience}/{self.cfg.PATIENCE}')

            if self.patience >= self.cfg.PATIENCE:
                print(f'\n{prefix}Early stopping at epoch {ep}')
                break
        print(f'\n{prefix}Done! Best Val WER: {self.best_wer:.2%}')
        return self.hist

## 13. Stage 1 — Main Training (50 epochs)

Standard hybrid CTC/Attention training with full augmentation.

In [ ]:
trainer = HybridTrainer(model, train_loader, val_loader, config, idx2word, SOS_IDX, EOS_IDX, DECODER_VOCAB_SIZE)
history_s1 = trainer.train(config.NUM_EPOCHS, stage_name="Stage 1")

## 14. Stage 2 — Fine-Tuning (reduced CTC weight + lower LR + mild augmentation)

Load best Stage 1 checkpoint, then fine-tune with:
- CTC weight reduced from 0.3 to 0.1 (CTC overfits 3.5x more than decoder)
- Lower learning rate (5e-5)
- Reduced augmentation intensity (50%)

In [ ]:
# Load best stage 1 model
ckpt = torch.load('best_model.pth', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best Stage 1 model (epoch {ckpt['epoch']}, WER: {ckpt['val_wer']:.2%})")

# Create new dataset + loader with mild augmentation
train_dataset_s2 = SignLanguageDataset(pose_data, train_labels, word2idx, normalizer, augmentor_mild, True)
train_loader_s2 = DataLoader(train_dataset_s2, config.BATCH_SIZE, True, collate_fn=collate_fn, num_workers=0, pin_memory=True)

# Stage 2 trainer with reduced CTC weight + lower LR
trainer_s2 = HybridTrainer(
    model, train_loader_s2, val_loader, config, idx2word, SOS_IDX, EOS_IDX, DECODER_VOCAB_SIZE,
    ctc_weight=config.STAGE2_CTC_WEIGHT,
    lr=config.STAGE2_LR,
    min_lr=config.STAGE2_MIN_LR,
    warmup=config.STAGE2_WARMUP_EPOCHS,
    num_epochs=config.STAGE2_EPOCHS,
)
# Carry over best WER from stage 1 so we only save if we actually improve
trainer_s2.best_wer = ckpt['val_wer']

history_s2 = trainer_s2.train(config.STAGE2_EPOCHS, stage_name="Stage 2")

## 15. Load Best Model + Training Visualization

In [ ]:
# Load the best model (could be from stage 1 or stage 2)
ckpt = torch.load('best_model.pth', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best model from {ckpt.get('stage', 'unknown')} (WER: {ckpt['val_wer']:.2%})")

# Combine histories
history = {}
for k in history_s1:
    history[k] = history_s1[k] + history_s2.get(k, [])

def plot_history(h):
    fig, ax = plt.subplots(2, 3, figsize=(18,10))
    s1_end = len(history_s1['train_loss'])
    ax[0,0].plot(h['train_loss'], label='Train'); ax[0,0].plot(h['val_loss'], label='Val')
    ax[0,0].axvline(s1_end-0.5, color='r', ls='--', alpha=0.5, label='Stage 2 start')
    ax[0,0].set_title('Total Loss'); ax[0,0].legend()
    ax[0,1].plot(h['train_ctc'], label='Train'); ax[0,1].plot(h['val_ctc'], label='Val')
    ax[0,1].axvline(s1_end-0.5, color='r', ls='--', alpha=0.5)
    ax[0,1].set_title('CTC Loss'); ax[0,1].legend()
    ax[0,2].plot(h['train_ce'], label='Train'); ax[0,2].plot(h['val_ce'], label='Val')
    ax[0,2].axvline(s1_end-0.5, color='r', ls='--', alpha=0.5)
    ax[0,2].set_title('CE Loss'); ax[0,2].legend()
    ax[1,0].plot([w*100 for w in h['train_wer']], label='Train'); ax[1,0].plot([w*100 for w in h['val_wer']], label='Val')
    ax[1,0].axvline(s1_end-0.5, color='r', ls='--', alpha=0.5)
    ax[1,0].set_title('WER (%)'); ax[1,0].set_ylabel('%'); ax[1,0].legend()
    ax[1,1].plot(h['lr'], color='g'); ax[1,1].set_title('LR'); ax[1,1].set_yscale('log')
    ax[1,1].axvline(s1_end-0.5, color='r', ls='--', alpha=0.5)
    g = [(v-t)*100 for t,v in zip(h['train_wer'], h['val_wer'])]
    ax[1,2].plot(g, color='r'); ax[1,2].set_title('WER Gap (Val-Train)'); ax[1,2].axhline(0, color='k', ls='--', alpha=.3)
    ax[1,2].axvline(s1_end-0.5, color='r', ls='--', alpha=0.5)
    for a in ax.flat: a.set_xlabel('Epoch'); a.grid(True, alpha=.3)
    plt.suptitle('Two-Stage Training (red line = Stage 2 start)', fontsize=14)
    plt.tight_layout(); plt.savefig('training_curves.png', dpi=150, bbox_inches='tight'); plt.show()
    be = np.argmin(h['val_wer'])+1; print(f'Best Val WER: {min(h["val_wer"]):.2%} at epoch {be}')

plot_history(history)

## 16. Length Penalty Sweep

Find the optimal length penalty alpha for AR beam search by evaluating on validation set.

In [ ]:
@torch.no_grad()
def evaluate_ar_beam_with_alpha(model, val_loader, idx2word, sos, eos, beam_width, max_len, alpha):
    """Run AR beam search on full val set with a specific length penalty alpha."""
    model.eval()
    refs, hyps, ids = [], [], []
    for inp, tgt, il, tl, sids in val_loader:
        inp = inp.to(device); il = il.to(device)
        enc, emask = model.encode(inp, il)
        preds = ar_beam_search(model, enc, emask, idx2word, sos, eos, beam_width, max_len, alpha)
        for j in range(len(inp)):
            hyps.append(preds[j])
            refs.append([idx2word[t.item()] for t in tgt[j,:tl[j].item()]])
            ids.append(sids[j])
    wer, _, _ = calc_wer(refs, hyps)
    return wer, refs, hyps, ids


print(f"Sweeping length penalty alpha over {config.LP_SWEEP_VALUES}")
print(f"AR Beam width: {config.AR_BEAM_WIDTH}")
print("=" * 60)

lp_results = {}
best_alpha = None
best_alpha_wer = float('inf')
best_alpha_data = None  # (refs, hyps, ids)

for alpha in config.LP_SWEEP_VALUES:
    wer, refs, hyps, ids = evaluate_ar_beam_with_alpha(
        model, val_loader, idx2word, SOS_IDX, EOS_IDX,
        config.AR_BEAM_WIDTH, config.MAX_DECODE_LEN, alpha
    )
    lp_results[alpha] = wer
    marker = ""
    if wer < best_alpha_wer:
        best_alpha_wer = wer
        best_alpha = alpha
        best_alpha_data = (refs, hyps, ids)
        marker = " <-- BEST"
    print(f"  alpha={alpha:.1f}  WER={wer:.2%}{marker}")

print("=" * 60)
print(f"Best alpha: {best_alpha} -> WER: {best_alpha_wer:.2%}")

# Plot the sweep
plt.figure(figsize=(8, 4))
alphas = sorted(lp_results.keys())
wers = [lp_results[a]*100 for a in alphas]
plt.plot(alphas, wers, 'bo-', linewidth=2, markersize=8)
plt.axvline(best_alpha, color='r', ls='--', alpha=0.7, label=f'Best: {best_alpha}')
plt.xlabel('Length Penalty Alpha'); plt.ylabel('WER (%)')
plt.title('Length Penalty Sweep'); plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout(); plt.savefig('lp_sweep.png', dpi=150, bbox_inches='tight'); plt.show()

## 17. Final Evaluation — All Methods Compared

Compare: CTC Beam | AR Greedy | AR Beam (best alpha from sweep)

In [ ]:
model.eval()

all_preds = {}   # method_name -> {sid: "gloss1 gloss2 ..."}
all_wers = {}    # method_name -> wer_float

# --- Method 1: CTC Beam Search ---
print("=" * 60)
print("Method 1: CTC Beam Search")
refs1, hyps1, ids1 = [], [], []
with torch.no_grad():
    for inp, tgt, il, tl, sids in tqdm(val_loader, desc='CTC Beam'):
        inp = inp.to(device); il = il.to(device)
        ctc_l, _, _ = model(inp, il)
        clp = F.log_softmax(ctc_l, -1)
        for j in range(len(inp)):
            hyps1.append(beam_search_ctc(clp[j,:il[j].item()], idx2word, config.BEAM_WIDTH))
            refs1.append([idx2word[t.item()] for t in tgt[j,:tl[j].item()]])
            ids1.append(sids[j])
wer1, _, _ = calc_wer(refs1, hyps1)
all_preds['CTC Beam Search'] = {sid: ' '.join(h) for sid, h in zip(ids1, hyps1)}
all_wers['CTC Beam Search'] = wer1
print(f"CTC Beam WER: {wer1:.2%}")

# --- Method 2: AR Greedy ---
print("\n" + "=" * 60)
print("Method 2: AR Greedy Decoding")
refs2, hyps2, ids2 = [], [], []
with torch.no_grad():
    for inp, tgt, il, tl, sids in tqdm(val_loader, desc='AR Greedy'):
        inp = inp.to(device); il = il.to(device)
        enc, emask = model.encode(inp, il)
        preds = ar_greedy_decode(model, enc, emask, idx2word, SOS_IDX, EOS_IDX, config.MAX_DECODE_LEN)
        for j in range(len(inp)):
            hyps2.append(preds[j])
            refs2.append([idx2word[t.item()] for t in tgt[j,:tl[j].item()]])
            ids2.append(sids[j])
wer2, _, _ = calc_wer(refs2, hyps2)
all_preds['AR Greedy'] = {sid: ' '.join(h) for sid, h in zip(ids2, hyps2)}
all_wers['AR Greedy'] = wer2
print(f"AR Greedy WER: {wer2:.2%}")

# --- Method 3: AR Beam Search (best alpha from sweep) ---
print("\n" + "=" * 60)
print(f"Method 3: AR Beam Search (width={config.AR_BEAM_WIDTH}, alpha={best_alpha})")
# Re-use results from the sweep (already computed)
refs3, hyps3, ids3 = best_alpha_data
all_preds['AR Beam Search'] = {sid: ' '.join(h) for sid, h in zip(ids3, hyps3)}
all_wers['AR Beam Search'] = best_alpha_wer
print(f"AR Beam WER: {best_alpha_wer:.2%}")

# === Dynamic Best Method Selection ===
print("\n" + "=" * 60)
print("FINAL RESULTS COMPARISON")
print("=" * 60)
for name, wer in sorted(all_wers.items(), key=lambda x: x[1]):
    print(f"  {name:30s} {wer:.2%}")

best_method = min(all_wers, key=all_wers.get)
best_wer = all_wers[best_method]
best_preds = all_preds[best_method]
print(f"\n  >>> BEST: {best_method} = {best_wer:.2%} <<<")

## 18. Sample Predictions

In [ ]:
@torch.no_grad()
def show_predictions(model, dataset, idx2word, n=15):
    model.eval()
    print("Sample Predictions")
    print("=" * 90)
    for i in range(min(n, len(dataset))):
        x, y, sid = dataset[i]
        T = len(x)
        x_dev = x.unsqueeze(0).to(device)
        il = torch.tensor([T], device=device)
        enc, emask = model.encode(x_dev, il)

        # CTC
        ctc_lp = F.log_softmax(model.ctc_head(enc), -1)
        p_ctc = beam_search_ctc(ctc_lp[0], idx2word, config.BEAM_WIDTH)

        # AR greedy
        p_ar_g = ar_greedy_decode(model, enc, emask, idx2word, SOS_IDX, EOS_IDX)[0]

        # AR beam (best alpha)
        p_ar_b = ar_beam_search(model, enc, emask, idx2word, SOS_IDX, EOS_IDX,
                                config.AR_BEAM_WIDTH, config.MAX_DECODE_LEN, best_alpha)[0]

        ref = [idx2word[t.item()] for t in y]
        print(f"\nSample {i} ({sid}):")
        print(f"  Ref:       {' '.join(ref)}")
        print(f"  CTC Beam:  {' '.join(p_ctc)}")
        print(f"  AR Greedy: {' '.join(p_ar_g)}")
        print(f"  AR Beam:   {' '.join(p_ar_b)}")
        w1 = levenshtein(ref, p_ctc)/max(len(ref),1)
        w2 = levenshtein(ref, p_ar_g)/max(len(ref),1)
        w3 = levenshtein(ref, p_ar_b)/max(len(ref),1)
        print(f"  WER: CTC={w1:.0%} AR_g={w2:.0%} AR_b={w3:.0%}")

show_predictions(model, val_dataset, idx2word, n=15)

## 19. Save Model + Generate Submission

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config.__dict__,
    'word2idx': word2idx, 'idx2word': idx2word,
    'history_s1': history_s1, 'history_s2': history_s2,
    'best_val_wer': min(trainer.best_wer, trainer_s2.best_wer),
    'best_method': best_method,
    'best_eval_wer': best_wer,
    'all_wers': all_wers,
    'best_alpha': best_alpha,
    'lp_results': lp_results,
    'sos_idx': SOS_IDX, 'eos_idx': EOS_IDX,
}, 'cslr_final_v3.pth')
print(f"Model saved. Best method: {best_method} ({best_wer:.2%})")
print(f"Best length penalty alpha: {best_alpha}")

In [ ]:
print(f"Generating submission with: {best_method} ({best_wer:.2%})")

pred_df = pd.DataFrame([{'id': k, 'gloss': v} for k, v in best_preds.items()])
pred_df.to_csv('predictions.csv', index=False)
print(f"Saved predictions.csv ({len(pred_df)} samples)")